# 📋 CL-SDRG Phase 1: Data Ingestion & Language Identification

**Script:** `01_data_ingestion_and_lid.py`  
**Target:** Google Colab T4 GPU (CPU-bound phase)  

This notebook:
1. Loads the FactCheck Insights (FCI) `claim_review.csv` (257k+ records)
2. Applies **Zero-Evidence-Leakage** sanitization (drops `ratingExplanation`)
3. Normalizes heterogeneous multilingual verdict labels → `{TRUE, FALSE, MIXED}`
4. Runs **FastText lid.176** for language identification
5. Filters and tags South Asian language claims (Urdu, Hindi, Bengali)
6. Mines **silver cross-lingual pairs** (±3 day publication window)
7. Creates **time-aware splits** (train ≤ 2023, test ≥ 2024)
8. Checks **Smoke Test Gate S₁** (Urdu count ≥ 500)

---
**⏱ Estimated Runtime:** 5–10 minutes

## 1. Environment Setup

In [ ]:
!pip install -q fasttext pandas numpy tqdm

In [ ]:
# Download FastText language identification model (126 MB)
import os
os.makedirs('outputs', exist_ok=True)
if not os.path.exists('outputs/lid.176.bin'):
    !wget -q https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin -O outputs/lid.176.bin
    print('Downloaded lid.176.bin')
else:
    print('lid.176.bin already exists')

## 2. Upload Dataset

Upload your `claim_review.csv` file when prompted below.

In [ ]:
import os

# Option A: Upload from local machine
DATA_DIR = 'Fact Check Dataset'
CSV_PATH = os.path.join(DATA_DIR, 'claim_review.csv')

if not os.path.exists(CSV_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    from google.colab import files
    print('Upload claim_review.csv:')
    uploaded = files.upload()
    for fname, content in uploaded.items():
        with open(CSV_PATH, 'wb') as f:
            f.write(content)
        print(f'Saved: {CSV_PATH} ({len(content)/1024/1024:.1f} MB)')
else:
    print(f'Dataset found: {CSV_PATH}')

## 3. Configuration

In [ ]:
from pathlib import Path

# ── Paths ──
PROJECT_ROOT = Path('.')
DATA_DIR = PROJECT_ROOT / 'Fact Check Dataset'
CLAIM_REVIEW_CSV = DATA_DIR / 'claim_review.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
PROCESSED_DATA_DIR = OUTPUT_DIR / 'processed_data'
LOG_DIR = OUTPUT_DIR / 'logs'
FIGURE_DIR = OUTPUT_DIR / 'figures'
FASTTEXT_MODEL_PATH = OUTPUT_DIR / 'lid.176.bin'

for d in [PROCESSED_DATA_DIR, LOG_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Retained / Leakage columns ──
RETAINED_COLUMNS = [
    'claimReviewed', 'itemReviewed.author.name', 'datePublished',
    'reviewRating.alternateName', 'author.name', 'url',
]
LEAKAGE_COLUMNS = ['reviewRating.ratingExplanation', 'reviewRating.ratingValue']

# ── Verdict mapping ──
VERDICT_MAP = {
    'true': 'TRUE', 'mostly true': 'TRUE', 'correct': 'TRUE', 'accurate': 'TRUE',
    'verified': 'TRUE', 'fact': 'TRUE', 'confirmed': 'TRUE',
    'false': 'FALSE', 'mostly false': 'FALSE', 'pants on fire': 'FALSE',
    'pants on fire!': 'FALSE', 'fake': 'FALSE', 'incorrect': 'FALSE',
    'not true': 'FALSE', 'fabricated': 'FALSE', 'debunked': 'FALSE',
    'half true': 'MIXED', 'half-true': 'MIXED', 'mixture': 'MIXED',
    'partially true': 'MIXED', 'partially false': 'MIXED',
    'misleading': 'MIXED', 'unverified': 'MIXED', 'unproven': 'MIXED',
    'out of context': 'MIXED', 'missing context': 'MIXED',
    'exaggerated': 'MIXED', 'needs context': 'MIXED',
    'distorts the facts': 'MIXED',
    'falso': 'FALSE', 'verdadeiro': 'TRUE', 'enganoso': 'MIXED',
    'impreciso': 'MIXED', 'insustentável': 'FALSE',
    'falsa': 'FALSE', 'verdadero': 'TRUE', 'engañoso': 'MIXED', 'verdadera': 'TRUE',
    'falsch': 'FALSE', 'richtig': 'TRUE', 'teilweise falsch': 'MIXED',
    'irreführend': 'MIXED', 'unbelegt': 'MIXED',
    'झूठ': 'FALSE', 'सच': 'TRUE', 'भ्रामक': 'MIXED', 'फर्जी': 'FALSE',
    'جھوٹ': 'FALSE', 'سچ': 'TRUE', 'گمراہ کن': 'MIXED',
}
LABEL2ID = {'TRUE': 0, 'FALSE': 1, 'MIXED': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ── Language & split config ──
TARGET_LANGUAGES = {'ur', 'hi', 'bn', 'pa', 'sd', 'ta', 'te', 'ml', 'mr', 'gu', 'ne', 'si'}
PRIMARY_FOCUS_LANGUAGES = {'ur', 'hi', 'bn'}
LID_CONFIDENCE_THRESHOLD = 0.5
SILVER_PAIR_DAY_WINDOW = 3
MIN_URDU_CLAIMS_THRESHOLD = 500
TRAIN_CUTOFF_DATE = '2023-12-31'
TEST_START_DATE = '2024-01-01'
RANDOM_SEED = 42

print('✅ Configuration loaded')

## 4. Helper Functions

In [ ]:
import random, time, logging, warnings, sys
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
from datetime import timedelta
from contextlib import contextmanager

warnings.filterwarnings('ignore')
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Logging setup
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter('[%(asctime)s] %(levelname)-8s %(message)s', datefmt='%H:%M:%S'))
logger.addHandler(handler)

def fmt(n): return f'{n:,}'

@contextmanager
def timer(label):
    start = time.perf_counter()
    logging.info(f'⏱  {label} — started')
    yield
    elapsed = time.perf_counter() - start
    if elapsed < 60:
        logging.info(f'✅ {label} — completed in {elapsed:.1f}s')
    else:
        m, s = divmod(elapsed, 60)
        logging.info(f'✅ {label} — completed in {int(m)}m {s:.1f}s')

def normalize_verdict(raw_label):
    if not isinstance(raw_label, str) or not raw_label.strip():
        return None
    cleaned = raw_label.strip().lower()
    if cleaned in VERDICT_MAP:
        return VERDICT_MAP[cleaned]
    for key, value in VERDICT_MAP.items():
        if key in cleaned or cleaned in key:
            return value
    return None

print('✅ Helpers loaded')

## 5. Step 1.1 — Data Ingestion & Zero-Leakage Scrubbing

In [ ]:
with timer('Loading raw CSV'):
    df = pd.read_csv(str(CLAIM_REVIEW_CSV), encoding='utf-8', low_memory=False, on_bad_lines='skip')
    logging.info(f'Raw dataset: {fmt(len(df))} rows, {df.shape[1]} columns')

# ── Zero-Leakage Sanitization ──
with timer('Zero-leakage sanitization'):
    for col in LEAKAGE_COLUMNS:
        if col in df.columns:
            non_null = df[col].notna().sum()
            logging.info(f"  Dropping leakage column: '{col}' ({fmt(non_null)} non-null)")
            df.drop(columns=[col], inplace=True)

    commentary_cols = [c for c in df.columns if 'explanation' in c.lower() or 'ratingValue' in c]
    if commentary_cols:
        logging.info(f'  Dropping commentary columns: {commentary_cols}')
        df.drop(columns=[c for c in commentary_cols if c in df.columns], inplace=True)

# ── Retain columns ──
available_cols = [c for c in RETAINED_COLUMNS if c in df.columns]
missing_cols = set(RETAINED_COLUMNS) - set(available_cols)
if missing_cols:
    logging.warning(f'  Missing expected columns: {missing_cols}')
df = df[available_cols].copy()

# ── Drop missing ──
initial = len(df)
df.dropna(subset=['claimReviewed', 'reviewRating.alternateName'], inplace=True)
logging.info(f'  Dropped {fmt(initial - len(df))} rows with missing claim/verdict')

df['claimReviewed'] = df['claimReviewed'].astype(str).str.strip()
df = df[df['claimReviewed'].str.len() > 5]

logging.info(f'  Sanitized dataset: {fmt(len(df))} rows')

## 6. Verdict Normalization

In [ ]:
with timer('Verdict normalization'):
    df['verdict'] = df['reviewRating.alternateName'].apply(normalize_verdict)

    unmapped = df['verdict'].isna().sum()
    if unmapped > 0:
        raw_unmapped = df.loc[df['verdict'].isna(), 'reviewRating.alternateName'].value_counts().head(20)
        logging.warning(f'  Unmapped verdict labels ({fmt(unmapped)} rows):')
        for label, count in raw_unmapped.items():
            logging.warning(f"    '{label}': {count}")

    df = df[df['verdict'].notna()].copy()
    df['label_id'] = df['verdict'].map(LABEL2ID)

    dist = df['verdict'].value_counts()
    logging.info('  Label distribution:')
    for label, count in dist.items():
        logging.info(f'    {label}: {fmt(count)} ({count/len(df)*100:.1f}%)')

## 7. Step 1.2 — FastText Language Identification

In [ ]:
import fasttext
fasttext.FastText.eprint = lambda x: None

with timer('FastText language identification'):
    ft_model = fasttext.load_model(str(FASTTEXT_MODEL_PATH))

    languages, confidences = [], []
    for text in tqdm(df['claimReviewed'].values, desc='Language ID', unit='claim'):
        clean = str(text).replace('\n', ' ').replace('\r', ' ').strip()
        if not clean:
            languages.append('unknown')
            confidences.append(0.0)
            continue
        preds = ft_model.predict(clean, k=1)
        languages.append(preds[0][0].replace('__label__', ''))
        confidences.append(float(preds[1][0]))

    df['detected_lang'] = languages
    df['lang_confidence'] = confidences

    lang_dist = df['detected_lang'].value_counts().head(20)
    logging.info('  Top 20 detected languages:')
    for lang, count in lang_dist.items():
        logging.info(f'    {lang}: {fmt(count)} ({count/len(df)*100:.1f}%)')

del ft_model  # Free memory

## 8. South Asian Filtering & Silver Pair Mining

In [ ]:
# ── Tag South Asian claims ──
df['is_south_asian'] = df['detected_lang'].isin(TARGET_LANGUAGES)
df['is_primary_focus'] = df['detected_lang'].isin(PRIMARY_FOCUS_LANGUAGES)

sa_mask = df['is_south_asian'] & (df['lang_confidence'] >= LID_CONFIDENCE_THRESHOLD)
sa_df = df[sa_mask].copy()

logging.info(f'South Asian claims (conf ≥ {LID_CONFIDENCE_THRESHOLD}):')
for lang, count in sa_df['detected_lang'].value_counts().items():
    logging.info(f'  {lang}: {fmt(count)}')
logging.info(f'Total South Asian: {fmt(len(sa_df))}')

# ── Silver pair mining ──
with timer(f'Silver pair mining (±{SILVER_PAIR_DAY_WINDOW} day window)'):
    df['datePublished'] = pd.to_datetime(df['datePublished'], errors='coerce', utc=True)

    pair_df = df.dropna(subset=['datePublished', 'author.name']).copy()
    pair_df = pair_df[pair_df['author.name'].str.strip().str.len() > 0]

    pairs = []
    for org, group in tqdm(pair_df.groupby('author.name'), desc='Mining pairs', unit='org'):
        if len(group) < 2: continue
        if group['detected_lang'].nunique() < 2: continue

        group = group.sort_values('datePublished')
        dates = group['datePublished'].values
        langs = group['detected_lang'].values
        claims = group['claimReviewed'].values
        idxs = group.index.values

        for i in range(len(group)):
            for j in range(i+1, len(group)):
                diff = abs((dates[j] - dates[i]) / np.timedelta64(1, 'D'))
                if diff > SILVER_PAIR_DAY_WINDOW: break
                if langs[i] != langs[j]:
                    pairs.append({
                        'claim_a': claims[i], 'lang_a': langs[i],
                        'claim_b': claims[j], 'lang_b': langs[j],
                        'org': org, 'date_diff_days': round(diff, 1),
                    })

    silver_pairs = pd.DataFrame(pairs)
    logging.info(f'  Discovered {fmt(len(silver_pairs))} silver cross-lingual pairs')

    if len(silver_pairs) > 0:
        pair_langs = silver_pairs.apply(lambda r: f"{r['lang_a']}-{r['lang_b']}", axis=1).value_counts().head(10)
        logging.info('  Top language pairs:')
        for name, count in pair_langs.items():
            logging.info(f'    {name}: {count}')

## 9. Step 1.3 — Time-Aware Dataset Splits

In [ ]:
with timer('Time-aware splitting'):
    valid_dates = df['datePublished'].notna()
    logging.info(f'  Rows with valid dates: {fmt(valid_dates.sum())} / {fmt(len(df))}')
    df_dated = df[valid_dates].copy()

    train_cutoff = pd.Timestamp(TRAIN_CUTOFF_DATE, tz='UTC')
    test_start = pd.Timestamp(TEST_START_DATE, tz='UTC')

    train_df = df_dated[df_dated['datePublished'] <= train_cutoff].copy()
    test_df = df_dated[df_dated['datePublished'] >= test_start].copy()

    logging.info(f'\n  Train set (≤ {TRAIN_CUTOFF_DATE}): {fmt(len(train_df))}')
    for label, count in train_df['verdict'].value_counts().items():
        logging.info(f'    {label}: {fmt(count)} ({count/len(train_df)*100:.1f}%)')

    logging.info(f'\n  Test set (≥ {TEST_START_DATE}): {fmt(len(test_df))}')
    for label, count in test_df['verdict'].value_counts().items():
        logging.info(f'    {label}: {fmt(count)} ({count/len(test_df)*100:.1f}%)')

# ── Smoke Test Gate S₁ ──
urdu_count = (df['detected_lang'] == 'ur').sum()
if urdu_count >= MIN_URDU_CLAIMS_THRESHOLD:
    logging.info(f'\n  ✅ Smoke Test S₁ PASSED: {fmt(urdu_count)} Urdu claims (≥ {MIN_URDU_CLAIMS_THRESHOLD})')
else:
    logging.warning(f'\n  ⚠️  Smoke Test S₁ FAILED: {fmt(urdu_count)} Urdu claims (need ≥ {MIN_URDU_CLAIMS_THRESHOLD})')
    logging.warning('  → NLLB-200 translation fallback required')

## 10. Save Processed Datasets

In [ ]:
with timer('Saving processed datasets'):
    df.to_csv(str(PROCESSED_DATA_DIR / 'fci_processed_full.csv'), index=False, encoding='utf-8')
    logging.info(f'  fci_processed_full.csv ({fmt(len(df))} rows)')

    train_df.to_csv(str(PROCESSED_DATA_DIR / 'fci_train.csv'), index=False, encoding='utf-8')
    logging.info(f'  fci_train.csv ({fmt(len(train_df))} rows)')

    test_df.to_csv(str(PROCESSED_DATA_DIR / 'fci_test.csv'), index=False, encoding='utf-8')
    logging.info(f'  fci_test.csv ({fmt(len(test_df))} rows)')

    sa_df.to_csv(str(PROCESSED_DATA_DIR / 'fci_south_asian.csv'), index=False, encoding='utf-8')
    logging.info(f'  fci_south_asian.csv ({fmt(len(sa_df))} rows)')

    if len(silver_pairs) > 0:
        silver_pairs.to_csv(str(PROCESSED_DATA_DIR / 'silver_pairs.csv'), index=False, encoding='utf-8')
        logging.info(f'  silver_pairs.csv ({fmt(len(silver_pairs))} pairs)')

    # South Asian splits
    sa_train = train_df[train_df['is_south_asian']]
    sa_test = test_df[test_df['is_south_asian']]
    sa_train.to_csv(str(PROCESSED_DATA_DIR / 'fci_sa_train.csv'), index=False, encoding='utf-8')
    sa_test.to_csv(str(PROCESSED_DATA_DIR / 'fci_sa_test.csv'), index=False, encoding='utf-8')
    logging.info(f'  SA train: {fmt(len(sa_train))}, SA test: {fmt(len(sa_test))}')

## 11. Final Summary

In [ ]:
print('\n' + '='*70)
print('  PHASE 1 COMPLETE — Summary Statistics')
print('='*70)
print(f'  Total raw records:          257,877')
print(f'  After sanitization:         {fmt(len(df))}')
print(f'  Unique languages detected:  {df["detected_lang"].nunique()}')
print(f'  South Asian claims:         {fmt(len(sa_df))}')
print(f'  Silver cross-lingual pairs: {fmt(len(silver_pairs))}')
print(f'  Train split:                {fmt(len(train_df))}')
print(f'  Test split:                 {fmt(len(test_df))}')
print(f'  Urdu claims:                {fmt(urdu_count)}')
print(f'  NLLB fallback needed:       {urdu_count < MIN_URDU_CLAIMS_THRESHOLD}')
print('='*70)
print('\n📋 Copy the output above and share it back for Phase 2 planning.')